<a href="https://colab.research.google.com/github/TAUforPython/LLM_AI_agents/blob/main/lesson-10_TAO_think_act_observe_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TAO (Think-Action-Observation) Pattern with Mistral LLM

TAO (Think-Action-Observation) Pattern:
- Think: Plan and reason about the next step
- Action: Execute a specific action/tool
- Observation: Observe the result and incorporate it into future reasoning



In [5]:
# Install required packages
!pip install langchain-mistralai langchain-core langchain-community -q

In [10]:


import os
from typing import List, Dict, Any
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
import json


In [8]:
# Configure Mistral API
MODEL_NAME = 'mistral-small-latest' # or "mistral-large-latest"

import os
from google.colab import userdata

os.environ["MISTRAL_API_KEY"] = userdata.get("Mistral_API")

llm = ChatMistralAI(
    model=MODEL_NAME,
    temperature=0,
    max_retries=2,
)

In [11]:

# Medical appointment database simulation
MEDICAL_DB = {
    "patients": {
        "P001": {"name": "John Doe", "age": 45, "conditions": ["hypertension"], "contact": "john@example.com"},
        "P002": {"name": "Jane Smith", "age": 32, "conditions": ["diabetes"], "contact": "jane@example.com"},
        "P003": {"name": "Robert Johnson", "age": 67, "conditions": ["arthritis", "cardiac issues"], "contact": "robert@example.com"}
    },
    "doctors": {
        "D001": {"name": "Dr. Sarah Williams", "specialty": "Cardiology", "schedule": {"2026-04-20": ["09:00", "10:00", "11:00"]}},
        "D002": {"name": "Dr. Michael Chen", "specialty": "Endocrinology", "schedule": {"2026-04-20": ["09:30", "10:30", "14:00"]}},
        "D003": {"name": "Dr. Emma Davis", "specialty": "Orthopedics", "schedule": {"2026-04-20": ["11:00", "13:00", "15:00"]}}
    },
    "appointments": {
        "A001": {"patient_id": "P001", "doctor_id": "D001", "date": "2026-04-20", "time": "09:00", "status": "confirmed"},
        "A002": {"patient_id": "P002", "doctor_id": "D002", "date": "2026-04-20", "time": "10:30", "status": "confirmed"}
    }
}

# Define tools for the TAO pattern
@tool
def get_patient_info(patient_id: str) -> str:
    """Get patient information by ID."""
    patient = MEDICAL_DB["patients"].get(patient_id)
    if patient:
        return json.dumps(patient)
    return json.dumps({"error": f"Patient {patient_id} not found"})

@tool
def get_doctor_schedule(doctor_id: str, date: str) -> str:
    """Get doctor's available time slots for a specific date."""
    doctor = MEDICAL_DB["doctors"].get(doctor_id)
    if doctor:
        available_slots = doctor["schedule"].get(date, [])
        return json.dumps({
            "doctor_name": doctor["name"],
            "specialty": doctor["specialty"],
            "date": date,
            "available_slots": available_slots
        })
    return json.dumps({"error": f"Doctor {doctor_id} not found"})

@tool
def book_appointment(patient_id: str, doctor_id: str, date: str, time: str) -> str:
    """Book an appointment for a patient with a doctor."""
    # Check if the slot is available
    doctor = MEDICAL_DB["doctors"].get(doctor_id)
    if not doctor:
        return json.dumps({"error": f"Doctor {doctor_id} not found"})

    if time not in doctor["schedule"].get(date, []):
        return json.dumps({"error": f"Time {time} on {date} is not available for {doctor['name']}"})

    # Check if there's already an appointment at this time
    for appt_id, appt in MEDICAL_DB["appointments"].items():
        if appt["date"] == date and appt["time"] == time and appt["doctor_id"] == doctor_id:
            return json.dumps({"error": f"Slot {time} on {date} is already booked"})

    # Create new appointment
    new_appt_id = f"A{len(MEDICAL_DB['appointments']) + 1:03d}"
    MEDICAL_DB["appointments"][new_appt_id] = {
        "patient_id": patient_id,
        "doctor_id": doctor_id,
        "date": date,
        "time": time,
        "status": "confirmed"
    }

    patient = MEDICAL_DB["patients"][patient_id]["name"]
    doctor_name = doctor["name"]
    return json.dumps({
        "success": True,
        "appointment_id": new_appt_id,
        "message": f"Appointment booked for {patient} with {doctor_name} on {date} at {time}"
    })

@tool
def get_appointment_info(appt_id: str) -> str:
    """Get appointment information by ID."""
    appt = MEDICAL_DB["appointments"].get(appt_id)
    if appt:
        patient = MEDICAL_DB["patients"][appt["patient_id"]]
        doctor = MEDICAL_DB["doctors"][appt["doctor_id"]]
        return json.dumps({
            "appointment_id": appt_id,
            "patient": patient["name"],
            "doctor": doctor["name"],
            "date": appt["date"],
            "time": appt["time"],
            "status": appt["status"]
        })
    return json.dumps({"error": f"Appointment {appt_id} not found"})

# Part 1: Chain-of-Thought (Pure reasoning without actions)
def chain_of_thought_approach(query: str) -> str:
    """
    Pure Chain-of-Thought: Only reasoning, no external actions
    """
    prompt = f"""
    Question: {query}

    Let's think step by step:
    1. What is the main question being asked?
    2. What information do we need to answer it?
    3. What would be the logical steps to reach the answer?

    Answer:
    """

    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)

    return response.content

# Part 2: Reasoning + Acting (TAO Pattern)
def reasoning_acting_approach(query: str) -> str:
    """
    TAO (Think-Action-Observation) Pattern:
    - Think: Reason about what to do next
    - Action: Execute a specific tool
    - Observation: Process the result
    """
    # Define the tools
    tools = [get_patient_info, get_doctor_schedule, book_appointment, get_appointment_info]

    # Create an agent that follows TAO pattern
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a medical appointment assistant. Follow the TAO (Think-Action-Observation) pattern:
        1. THINK: Analyze the user's request and plan your approach
        2. ACTION: Use the appropriate tool to get information or perform a task
        3. OBSERVATION: Process the result and decide next steps

        Available tools:
        - get_patient_info: Get patient information by ID
        - get_doctor_schedule: Get doctor's available time slots
        - book_appointment: Book an appointment
        - get_appointment_info: Get appointment details by ID"""),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agent = create_tool_calling_agent(llm, tools, prompt)
    agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

    response = agent_executor.invoke({"input": query})
    return response["output"]

# Part 3: Reasoning (o1-style) - Deep reasoning with multiple steps
def o1_style_reasoning(query: str) -> str:
    """
    o1-style reasoning: Deep, multi-step reasoning with exploration of possibilities
    """
    prompt = f"""
    Question: {query}

    Let's engage in deep, thorough reasoning to explore multiple angles and possibilities:

    Initial Analysis:
    - What is the core request?
    - What are the implicit requirements?
    - What could be potential complications?

    Exploring Possibilities:
    - What are different ways to approach this?
    - What information might be missing?
    - What assumptions should we verify?

    Step-by-Step Solution:
    - Break down the solution into logical steps
    - Consider edge cases and alternatives
    - Validate each step for completeness

    Final Synthesis:
    - Summarize the approach
    - Highlight key considerations
    - Provide the final answer

    Begin the reasoning process:
    """

    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)

    return response.content

# Demonstration
print("TAO (Think-Action-Observation) PATTERN DEMONSTRATION")
print("=" * 60)

# Example queries
queries = [
    "Book an appointment for John Doe with a cardiologist on April 20, 2026",
    "What are the available time slots for Dr. Michael Chen on April 20, 2026?",
    "Get information about appointment A001"
]

print("\n1. CHAIN OF THOUGHT APPROACH")
print("-" * 40)
for i, query in enumerate(queries[:1], 1):
    print(f"Query {i}: {query}")
    cot_response = chain_of_thought_approach(query)
    print(f"Response: {cot_response[:200]}...")
    print()

print("\n2. REASONING + ACTING (TAO PATTERN)")
print("-" * 40)
for i, query in enumerate(queries, 1):
    print(f"Query {i}: {query}")
    try:
        ta_response = reasoning_acting_approach(query)
        print(f"Response: {ta_response}")
    except Exception as e:
        print(f"Error: {e}")
    print()

print("\n3. O1-STYLE REASONING")
print("-" * 40)
for i, query in enumerate(queries[:1], 1):
    print(f"Query {i}: {query}")
    o1_response = o1_style_reasoning(query)
    print(f"Response: {o1_response[:200]}...")
    print()

print("\n" + "=" * 60)
print("TAO PATTERN ANALYSIS:")
print("• Chain-of-Thought: Pure reasoning, no external interaction")
print("• TAO (Reasoning+Acting): Combines thinking with tool usage")
print("• O1-Style: Deep, exploratory reasoning with multiple perspectives")
print("=" * 60)

# Show the database state after TAO interactions
print("\nDATABASE STATE AFTER INTERACTIONS:")
print("Patients:", list(MEDICAL_DB["patients"].keys()))
print("Doctors:", list(MEDICAL_DB["doctors"].keys()))
print("Appointments:", {k: f"{v['patient_id']} with {v['doctor_id']} at {v['date']} {v['time']}" for k, v in MEDICAL_DB["appointments"].items()})

TAO (Think-Action-Observation) PATTERN DEMONSTRATION

1. CHAIN OF THOUGHT APPROACH
----------------------------------------
Query 1: Book an appointment for John Doe with a cardiologist on April 20, 2026
Response: 1. **What is the main question being asked?**
   The user wants to book an appointment for "John Doe" with a cardiologist on April 20, 2026.

2. **What information do we need to answer it?**
   - The ...


2. REASONING + ACTING (TAO PATTERN)
----------------------------------------
Query 1: Book an appointment for John Doe with a cardiologist on April 20, 2026


> Entering new AgentExecutor chain...

Invoking: `get_patient_info` with `{'patient_id': 'John Doe'}`


{"error": "Patient John Doe not found"}
Invoking: `get_patient_info` with `{'patient_id': '12345'}`


{"error": "Patient 12345 not found"}
Invoking: `get_patient_info` with `{'patient_id': '67890'}`


{"error": "Patient 67890 not found"}
Invoking: `get_patient_info` with `{'patient_id': 'John Doe'}`
responded: THINK